In [ ]:
import pandas as pd
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from lifelines import KaplanMeierFitter
import os
import glob
import xgboost as xgb
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test
from sksurv.metrics import brier_score
from sksurv.util import Surv
from sksurv.metrics import integrated_brier_score
import shap
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from adjustText import adjust_text

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.pipeline.constants import EVENT_COL, DURATION_COL, ORIGINAL_MEASUREMENT_COLS

In [ ]:
json_path: str = "/Users/marekpuskas/Documents/Diplomovka/DP-main/output/09-02-2026_12:40:16_xgb_aft_positive_imputer_10_times/final_aggregated_summary/results.json"
with open(file=json_path, mode="r", encoding="utf-8") as file:
    summary = json.load(file)

In [ ]:
waves = ["wave_1", "wave_4"]

for wave in waves:
    train_c = summary[wave]["all_train_c_indices"]
    test_c = summary[wave]["all_test_c_indices"]

    lowest_diff: float = float("inf")
    best_run: int = 1

    for c in range(len(train_c)):
        diff = train_c[c] - test_c[c]
        if diff < 0:
            print(f"diff of train and test is less than 0")
            continue
        if diff < lowest_diff:
            lowest_diff = diff
            best_run = c+1
    print(f"Wave {wave[-1]}")
    print(f"Lowest diff run: {best_run}")

In [ ]:
xgb_1 = pd.read_pickle("/Users/marekpuskas/Documents/Diplomovka/DP-main/output/26-01-2026_20:59:35_xgb_aft_10_times/wave_1_run_3_model_xgb_aft/model.pkl")
xgb_4 = pd.read_pickle("/Users/marekpuskas/Documents/Diplomovka/DP-main/output/26-01-2026_20:59:35_xgb_aft_10_times/wave_4_run_4_model_xgb_aft/model.pkl")

rsf_1 = pd.read_pickle("/Users/marekpuskas/Documents/Diplomovka/DP-main/output/26-01-2026_21:16:12_rsf_all_10_times/wave_1_run_9_model_rsf/model.pkl")
rsf_4 = pd.read_pickle("/Users/marekpuskas/Documents/Diplomovka/DP-main/output/26-01-2026_21:16:12_rsf_all_10_times/wave_4_run_10_model_rsf/model.pkl")

In [ ]:
imputed_w1 = pd.read_pickle("/Users/marekpuskas/Documents/Diplomovka/DP-main/DATA/imputed_cache/wave_1_imputed.parquet")
imputed_w4 = pd.read_pickle("/Users/marekpuskas/Documents/Diplomovka/DP-main/DATA/imputed_cache/wave_4_imputed.parquet")

In [ ]:
imputed_train_w1, imputed_test_w1 = imputed_w1["splits"]
imputed_train_w4, imputed_test_w4 = imputed_w4["splits"]

In [ ]:
imputed_test_w1

In [ ]:
def calculate_aic(actual_model, X, y_duration, y_event, feature_names):
    """
    Vypočíta AIC pre XGBoost AFT model so správnym nastavením AFT labelov.
    """
    k = len(feature_names)
    y_lower = y_duration
    y_upper = np.where(y_event == 1, y_duration, np.inf)

    dmat = xgb.DMatrix(X)
    dmat.set_float_info('label_lower_bound', y_lower)
    dmat.set_float_info('label_upper_bound', y_upper)
    
    booster = actual_model if isinstance(actual_model, xgb.Booster) else actual_model.get_booster()
    
    try:
        eval_result = booster.eval(dmat)
        nloglik = float(eval_result.split(':')[1])
        
        log_L = -nloglik * X.shape[0]

        aic = 2*k - 2*log_L
        return aic
    except Exception as e:
        print(f"Detailná chyba pri eval: {e}")
        return np.nan

def compare_models_stability_vs_aic(base_path, wave_num, data, feature_cols, duration_col, event_col):
    results = []
    search_pattern = os.path.join(base_path, f"wave_{wave_num}_run_*_model_xgb_aft/model.pkl")
    model_paths = glob.glob(search_pattern)
    
    if not model_paths:
        print(f"Upozornenie: Nenašli sa modely pre vlnu {wave_num}.")
        return pd.DataFrame()
    wave_key = f"wave_{wave_num}"

    for path in model_paths:
        run_name = path.split('/')[-2]
        try:
            wrapper = pd.read_pickle(path)
            actual_model = wrapper.model if hasattr(wrapper, 'model') else wrapper

            run_number = int(run_name.split('_')[3])
            list_index = run_number - 1 

            c_index_train = summary[wave_key]["all_train_c_indices"][list_index]*100
            c_index_test = summary[wave_key]["all_test_c_indices"][list_index]*100
            c_gap = c_index_train - c_index_test

            aic_val = calculate_aic(
                actual_model, 
                data[feature_cols], 
                data[duration_col], 
                data[event_col], 
                feature_cols
            )
            
            results.append({
                "Run": run_name,
                "Train C-index": round(c_index_train, 2),
                "Test C-index": round(c_index_test, 2),
                "Gap": round(c_gap, 2),
                "AIC": round(aic_val, 2)
            })
        except Exception as e:
            print(f"Nepodarilo sa spracovať {run_name}: {e}")

    if not results:
        print(f"Chyba: Zoznam výsledkov pre vlnu {wave_num} je prázdny. Skontrolujte vypísané chyby.")
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values("Run").reset_index(drop=True)

path_to_runs = "/Users/marekpuskas/Documents/Diplomovka/DP-main/output/26-01-2026_20:59:35_xgb_aft_10_times/"

print("-" * 30)
print("POROVNANIE MODELOV: VLNA 1")
df_w1 = compare_models_stability_vs_aic(path_to_runs, 1, imputed_test_w1, ORIGINAL_MEASUREMENT_COLS, DURATION_COL, EVENT_COL)
print(df_w1)

print("\n" + "-" * 30)
print("POROVNANIE MODELOV: VLNA 4")
df_w4 = compare_models_stability_vs_aic(path_to_runs, 4, imputed_test_w4, ORIGINAL_MEASUREMENT_COLS, DURATION_COL, EVENT_COL)
print(df_w4)

In [ ]:
def plot_xgb_km_risk_subplots(
    xgb_w1, xgb_w4,
    data_w1, data_w4,
    feature_cols
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False) 
    plt.subplots_adjust(wspace=0.25)

    for ax, model, data, title in [
        (axes[0], xgb_w1, data_w1, "XGB AFT - risk stratification (Wave 1)"),
        (axes[1], xgb_w4, data_w4, "XGB AFT - risk stratification (Wave 4)")
    ]:
        preds = model.predict_risk(data[feature_cols])
        median_risk = np.median(preds)

        high_mask = preds <= median_risk
        low_mask = ~high_mask

        deaths_high = data.loc[high_mask, EVENT_COL].sum()
        total_high = len(data.loc[high_mask])
        
        deaths_low = data.loc[low_mask, EVENT_COL].sum()
        total_low = len(data.loc[low_mask])

        results = logrank_test(
            data.loc[high_mask, DURATION_COL],
            data.loc[low_mask, DURATION_COL],
            event_observed_A=data.loc[high_mask, EVENT_COL],
            event_observed_B=data.loc[low_mask, EVENT_COL]
        )
        p_val = results.p_value
        p_text = f"p < 0.001" if p_val < 0.001 else f"p = {p_val:.4f}"
        
        kmf = KaplanMeierFitter()

        kmf.fit(
            data.loc[high_mask, DURATION_COL],
            data.loc[high_mask, EVENT_COL],
            label="High Risk"
        )
        kmf.plot_survival_function(ax=ax, color="red", linewidth=2)

        kmf.fit(
            data.loc[low_mask, DURATION_COL],
            data.loc[low_mask, EVENT_COL],
            label="Low Risk"
        )
        kmf.plot_survival_function(ax=ax, color="green", linewidth=2)

        ax.text(0.025, 0.31, f"Log-rank test:\n{p_text}", transform=ax.transAxes, 
                fontsize=11, verticalalignment='bottom', 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5, edgecolor='gray'))

        info_text = (
            f"Number of Deaths\n"
            f"High Risk: {int(deaths_high)}\n"
            f"Low Risk: {int(deaths_low)}"
        )
        ax.text(0.024, 0.165, info_text, transform=ax.transAxes, 
                fontsize=11, verticalalignment='bottom', 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='gray'))

        ax.set_title(title, fontsize=18, pad=20)
        ax.set_xlabel("Time (days)", fontsize=16)
        ax.set_ylabel("Survival Probability", fontsize=16)

        ax.set_ylim(0.5, 1.02)
        ax.tick_params(axis="x", labelsize=14)
        ax.tick_params(axis="y", labelsize=14)

        ax.legend(loc="lower left", frameon=True, fontsize=12)
        ax.grid(alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.show()

plot_xgb_km_risk_subplots(
    xgb_w1=xgb_1,
    xgb_w4=xgb_4,
    data_w1=imputed_test_w1,
    data_w4=imputed_test_w4,
    feature_cols=ORIGINAL_MEASUREMENT_COLS
)

In [ ]:
wave1_color  = "#2E76C4"
wave4_color  = "#bfb34b"
death1_color = "#104a7d"
death4_color = "#716E29"

def plot_rsf_vs_km_subplots(
    rsf_w1, rsf_w4,
    data_w1, data_w4,
    feature_cols
):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    plt.subplots_adjust(wspace=0.25)

    for ax, model, data, title in [
        (axes[0], rsf_w1, data_w1, "RSF vs KM (Wave 1)"),
        (axes[1], rsf_w4, data_w4, "RSF vs KM (Wave 4)")
    ]:
        surv_funcs = model.model.predict_survival_function(
            data[feature_cols]
        )

        max_t = min(f.domain[1] for f in surv_funcs)
        times = np.arange(0, max_t + 1)

        mean_surv = np.mean(
            [f(times) for f in surv_funcs],
            axis=0,
        )

        ax.step(
            times,
            mean_surv,
            where="post",
            label="RSF - average prediction",
            lw=2,
            color=wave1_color if "Wave 1" in title else wave4_color
        )

        kmf = KaplanMeierFitter()
        kmf.fit(
            data[DURATION_COL],
            data[EVENT_COL],
            label="Kaplan-Meier (ground truth)"
        )
        kmf.plot_survival_function(
            ax=ax,
            linestyle="--",
            color="red"
        )

        ax.set_title(title, fontsize=16)
        ax.set_xlabel(xlabel="Time (days)", fontsize=14)
        ax.grid(alpha=0.3, linestyle="--")

        ax.legend(
            loc="lower left",
            frameon=True,
            fontsize=12,
        )

        ax.set_ylabel(ylabel="Survival Probability", fontsize=14)

        ax.set_ylim(0.5, 1.02)
        ax.set_xlim(0, 35)

        ax.tick_params(axis="x", labelsize=14)
        ax.tick_params(axis="y", labelsize=14)

    plt.show()

plot_rsf_vs_km_subplots(
    rsf_w1=rsf_1,
    rsf_w4=rsf_4,
    data_w1=imputed_test_w1,
    data_w4=imputed_test_w4,
    feature_cols=ORIGINAL_MEASUREMENT_COLS
)

In [ ]:
def plot_model_error_in_time(rsf_w1, rsf_w4, train_w1, test_w1, train_w4, test_w4):
    plt.figure(figsize=(9, 5.5))

    for model, train, test, wave, col in [
        (rsf_w1, train_w1, test_w1, "Wave 1", wave1_color),
        (rsf_w4, train_w4, test_w4, "Wave 4", wave4_color)
    ]:
        y_train = Surv.from_dataframe(EVENT_COL, DURATION_COL, train)
        y_test = Surv.from_dataframe(EVENT_COL, DURATION_COL, test)

        surv_funcs = model.model.predict_survival_function(
            test[ORIGINAL_MEASUREMENT_COLS]
        )

        max_common_time = min(f.domain[1] for f in surv_funcs)

        candidate_times = np.percentile(
            test[DURATION_COL][test[DURATION_COL] <= max_common_time],
            np.linspace(0, 100, 150)
        )

        scores = []
        valid_times = []

        for t in candidate_times:
            try:
                preds_t = np.array([f(t) for f in surv_funcs]).reshape(-1, 1)
                _, s = brier_score(y_train, y_test, preds_t, [t])
                scores.append(s[0])
                valid_times.append(t)
            except ValueError:
                continue

        plt.plot(
            valid_times,
            scores,
            marker="o",
            lw=2,
            label=f"Brier Score - {wave}",
            color=col
        )

    plt.axhline(
        0.25,
        color="red",
        linestyle="--",
        label="Random Guess (0.25)"
    )

    plt.tick_params(axis="x", labelsize=10)
    plt.tick_params(axis="y", labelsize=10)


    plt.title("Brier Score of RSF over Time", fontsize=14)
    plt.xlabel("Time (days)", fontsize=12)
    plt.ylabel("Brier Score", fontsize=12)
    plt.ylim(0, 0.4)
    plt.legend()
    plt.grid(alpha=0.3, linestyle="--")
    plt.show()

plot_model_error_in_time(rsf_1, rsf_4, imputed_train_w1, imputed_test_w1, imputed_train_w4, imputed_test_w4)

In [ ]:
def calculate_ibs(model, train_df, test_df, feature_cols):
    """
    Vypočíta IBS pre model na testovacích dátach.
    """
    def to_structured(df):
        return np.array(
            [(bool(e), t) for e, t in zip(df[EVENT_COL], df[DURATION_COL])],
            dtype=[('event', 'bool'), ('time', 'float')]
        )

    y_train = to_structured(train_df)
    y_test = to_structured(test_df)

    max_time = y_test['time'].max()
    times = np.linspace(1, max_time - 1, 150)

    surv_funcs = model.model.predict_survival_function(test_df[feature_cols])
    preds = np.row_stack([f(times) for f in surv_funcs])

    ibs_value = integrated_brier_score(y_train, y_test, preds, times)
    
    return ibs_value

ibs_w1 = calculate_ibs(rsf_1, imputed_train_w1, imputed_test_w1, ORIGINAL_MEASUREMENT_COLS)
ibs_w4 = calculate_ibs(rsf_4, imputed_train_w4, imputed_test_w4, ORIGINAL_MEASUREMENT_COLS)

print(f"IBS pre Vlnu 1: {ibs_w1:.4f}")
print(f"IBS pre Vlnu 4: {ibs_w4:.4f}")

In [ ]:
moje_normy = {
    "APTT-R": (0.85, 1.15),
    "CD19+": (0.1, 0.5),
    "CD3+": (0.7, 2.1),
    "CD4+": (0.3, 1.4),
    "CD4+/CD8+": (1.1, 2.9),
    "CD8+": (0.2, 0.9),
    "D-dimér HS": (0.03, 0.5),
    "Eo abs": (0.05, 0.25),
    "Fib": (1.8, 3.5),
    "HGB": (12.0, 18.0),  # Zlúčené rozmedzie pre ženy aj mužov
    "Ly abs": (1.5, 4.0),
    "NE/LY(NLR)": (0.0, 2.99),
    "NK": (0.09, 0.6),
    "Neu abs": (1.4, 6.5),
    "P-Laktát": (0.5, 2.2),
    "PDW": (15.5, 17.1),
    "PLT": (150.0, 400.0),
    "PT (INR)": (0.85, 1.15),
    "S-ALP": (0.5, 2.0),
    "S-ALT": (0.05, 0.85),  # Horná hranica pre mužov
    "S-AST": (0.05, 0.85),  # Odhadnuté podľa ALT (v zozname chýbal mužský údaj)
    "S-Alb": (35.0, 52.0),
    "S-Bil-T": (5.0, 21.0),
    "S-CB": (66.0, 83.0),
    "S-CK": (0.33, 2.85),
    "S-CK-MB": (0.05, 0.40),
    "S-CL": (105.0, 105.0), # Fixná hodnota
    "S-CRP": (0.1, 5.0),
    "S-FER": (11.0, 307.0),
    "S-GMT": (0.05, 0.92),  # Horná hranica pre mužov
    "S-Gluk": (4.1, 5.9),
    "S-IL6": (1.5, 7.0),
    "S-K": (3.5, 5.1),
    "S-Kreat": (49.0, 104.0), # Rozsah pokrývajúci obe pohlavia
    "S-Na": (136.0, 146.0),
    "S-TnT": (0.003, 0.014),
    "S-PBNP": (5.0, 125.0),
    "S-Urea": (2.8, 7.2),
    "S-VITD": (20.0, 100.0),
    "SatO2 %": (95.0, 99.0),
    "WBC": (4.0, 10.0)
}

def plot_shap_dependence_grid(
    model,
    data,
    feature_cols,
    ref_intervals,
    n_features=12,
    n_rows=3,
    n_cols=3,
    figsize=(14, 12),
    lowess_frac=0.3,
    title=""
):
    explainer = shap.TreeExplainer(model.model)
    shap_values = explainer.shap_values(data[feature_cols])

    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:n_features]
    top_features = [feature_cols[i] for i in top_idx]

    global_y_min = np.nanmin(shap_values[:, top_idx])
    global_y_max = np.nanmax(shap_values[:, top_idx])

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=figsize,
        sharey=False
    )
    axes = axes.flatten()

    norm = Normalize(vmin=global_y_min, vmax=global_y_max)
    cmap = plt.cm.seismic_r

    for ax, feat in zip(axes, top_features):
        feat_idx = feature_cols.index(feat)

        x = data[feat].values
        y = shap_values[:, feat_idx]

        mask = ~np.isnan(x) & ~np.isnan(y)
        x = x[mask]
        y = y[mask]

        ax.scatter(
            x,
            y,
            c=y,
            cmap=cmap,
            norm=norm,
            s=50,
            alpha=0.65,
            linewidths=0.5,
            edgecolor='k'
        )

        ax.axhline(0, color="black", linestyle="--", linewidth=1.0)

        if len(x) > 20:
            order = np.argsort(x)
            smoothed = lowess(
                y[order],
                x[order],
                frac=lowess_frac,
                return_sorted=True
            )

            crossings = np.where(np.diff(np.sign(smoothed[:, 1])))[0]

            if len(crossings) > 0:
                idx_cross = crossings[0]

                x0 = smoothed[idx_cross, 0]
                x1 = smoothed[idx_cross + 1, 0]
                y0 = smoothed[idx_cross, 1]
                y1 = smoothed[idx_cross + 1, 1]

                if y1 != y0:
                    x_cross = x0 - y0 * (x1 - x0) / (y1 - y0)

                    ax.axvline(
                        x_cross,
                        color="red",
                        linestyle=":",
                        linewidth=1.5
                    )

                    ax.text(
                        x_cross,
                        -0.08,
                        f"{x_cross:.2f}",
                        transform=ax.get_xaxis_transform(),
                        ha="center",
                        va="top",
                        fontsize=10,
                        color="red"
                    )

        if feat in ref_intervals:
            low, high = ref_intervals[feat]
            if low is not None and high is not None and low != high:
                ax.axvspan(low, high, color="gray", alpha=0.2)

        ax.set_title(feat, fontsize=14, pad=10)
        ax.set_xlabel("Test Value", fontsize=12)
        ax.set_ylabel("SHAP (contribution to log(T))", fontsize=12)
        ax.tick_params(axis='x', labelsize=10)
        ax.tick_params(axis='y', labelsize=10)
        ax.grid(alpha=0.3, linestyle="--")

    for ax in axes[len(top_features):]:
        ax.axis("off")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("SHAP Value", fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    fig.suptitle(title, fontsize=18, y=0.94)
    fig.subplots_adjust(hspace=0.30, wspace=0.4, bottom=0.05)

    plt.show()


plot_shap_dependence_grid(
    model=xgb_1,
    data=imputed_test_w1,
    feature_cols=ORIGINAL_MEASUREMENT_COLS,
    ref_intervals=moje_normy,
    n_features=9,
    n_rows=3,
    n_cols=3,
    title="SHAP Dependence of the Most Important Tests - Wave 1"
)

plot_shap_dependence_grid(
    model=xgb_4,
    data=imputed_test_w4,
    feature_cols=ORIGINAL_MEASUREMENT_COLS,
    ref_intervals=moje_normy,
    n_features=9,
    n_rows=3,
    n_cols=3,
    title="SHAP Dependence of the Most Important Tests - Wave 4"
)

In [ ]:
def shap_importance_dataframe(model, data, feature_cols):
    explainer = shap.TreeExplainer(model.model)
    shap_values = explainer.shap_values(data[feature_cols])

    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

    return pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap
    })


def shap_stability_scatter(
    model_w1,
    model_w4,
    data_w1,
    data_w4,
    feature_cols,
    top_n=20
):
    df1 = shap_importance_dataframe(model_w1, data_w1, feature_cols)
    df4 = shap_importance_dataframe(model_w4, data_w4, feature_cols)

    df = df1.merge(
        df4,
        on="feature",
        suffixes=("_w1", "_w4")
    )

    df["importance_max"] = df[["mean_abs_shap_w1", "mean_abs_shap_w4"]].max(axis=1)
    df = df.sort_values("importance_max", ascending=False).head(top_n)


    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    ax.scatter(
        df["mean_abs_shap_w1"],
        df["mean_abs_shap_w4"],
        s=50,
        alpha=0.75,
        c=["red" if r["mean_abs_shap_w4"] > r["mean_abs_shap_w1"] else "blue" for _, r in df.iterrows()],
        edgecolors="black",
        linewidths=1.0,
    )

    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])
    ]
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    x_bg = np.linspace(lims[0], lims[1], 300)

    ax.fill_between(
        x_bg,
        lims[0],
        x_bg,
        color="blue",
        alpha=0.05,
        label="Vyššia dôležitosť - Vlna 1"
    )

    ax.fill_between(
        x_bg,
        x_bg,
        lims[1],
        color="red",
        alpha=0.05,
        label="Vyššia dôležitosť - Vlna 4"
    )

    ax.plot(lims, lims, "k--", linewidth=1.2, alpha=0.8)

    texts = []
    for _, r in df.iterrows():
        texts.append(
            ax.text(
                r["mean_abs_shap_w1"],
                r["mean_abs_shap_w4"],
                r["feature"],
                fontsize=9,
                ha="center",
                va="center",
                c="red" if r["mean_abs_shap_w4"] > r["mean_abs_shap_w1"] else "blue",
            )
        )

    threshold = 0.2

    for _, r in df.iterrows():
        x = r["mean_abs_shap_w1"]
        y = r["mean_abs_shap_w4"]

        if max(x, y) < threshold:
            continue

        ax.plot(
            [x, x], [lims[0], y],
            linestyle="--",
            linewidth=0.75,
            color="black",
            alpha=0.3,
            zorder=0
        )

        ax.plot(
            [lims[0], x], [y, y],
            linestyle="--",
            linewidth=0.75,
            color="black",
            alpha=0.3,
            zorder=0
        )

        ax.text(
            x,
            lims[0],
            f"{x:.2f}",
            ha="center",
            va="top",
            fontsize=8.5,
            color="blue",
            alpha=0.85,
        )

        ax.text(
            lims[0],
            y,
            f"{y:.2f}",
            ha="right",
            va="center",
            fontsize=8.5,
            color="red",
            alpha=0.85,
        )


    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(
            arrowstyle="-",
            color="black",
            alpha=0.65,
            linewidth=0.5
        ),
        expand_points=(1.4, 1.4),
        expand_text=(1.4, 1.4),
        force_text=1.5
    )

    ax.set_xlabel("Priemerná absolútna SHAP dôležitosť - Vlna 1")
    ax.set_ylabel("Priemerná absolútna SHAP dôležitosť - Vlna 4")
    ax.set_title("Globálna dôležitosť atribútov medzi vlnami")

    ax.grid(alpha=0.3, linestyle="--")
    ax.legend(frameon=True, fontsize=9, loc="upper left")

    plt.tight_layout()
    plt.show()

shap_stability_scatter(
    model_w1=xgb_1,
    model_w4=xgb_4,
    data_w1=imputed_test_w1,
    data_w4=imputed_test_w4,
    feature_cols=ORIGINAL_MEASUREMENT_COLS,
    top_n=20
)

In [ ]:
def compute_shap_ranks(model, data, feature_cols):
    explainer = shap.TreeExplainer(model.model)
    shap_values = explainer.shap_values(data[feature_cols])

    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

    df = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap
    })

    df = df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    df["rank"] = df.index + 1

    return df

shap_rank_w1 = compute_shap_ranks(
    model=xgb_1,
    data=imputed_test_w1,
    feature_cols=ORIGINAL_MEASUREMENT_COLS
)

shap_rank_w4 = compute_shap_ranks(
    model=xgb_4,
    data=imputed_test_w4,
    feature_cols=ORIGINAL_MEASUREMENT_COLS
)

comparison = (
    shap_rank_w1
    .merge(
        shap_rank_w4,
        on="feature",
        how="inner",
        suffixes=("_w1", "_w4")
    )
)

comparison["rank_change"] = comparison["rank_w1"] - comparison["rank_w4"]

comparison = comparison.sort_values("rank_w1")

top_20 = comparison[
    (comparison["rank_w1"] <= 20) |
    (comparison["rank_w4"] <= 20)
]
top_20